# 🌾 Kaggriculture Agent — Capstone Writeup

**AI Agents: Intensive Vibe Coding Capstone Project (Google × Kaggle)**

An autonomous agent for **Kaggriculture** — a two-player farming game (30 days =
**720 turns**). It drives a **farmer** around a tile grid, one action per turn:
buy seeds, plant, water, pull weeds, harvest into a shed, and sell — aiming to
out-earn the opponent.

| | |
|---|---|
| **Stack** | Python (stdlib agent) + a local simulator + optional ADK/Gemini infra |
| **Approach** | Contract-first: built & tested against the real obs/action shapes without the Kaggle package |
| **Status** | Runs a full 720-turn game; 66 tests green; self-contained submission ready |
| **Video** | _paste link here_ |
| **Code** | _paste repo / notebook link here_ |

> **Honest note:** local scores below use *assumed* game constants against an
> *idle* opponent — a correctness/sanity signal, **not** a leaderboard result.
> The real environment supplies the real economics and only calls our `agent(obs)`.

## 1. The problem

Kaggriculture is **spatial and two-player**. Each turn the agent receives an
`obs` — its farmer's `(x, y)` position, every tile on the grid, its money, and
(privately) its seeds and shed — and must return **one** farmer command (a move,
or an action on the current tile) plus optional market orders.

The hard parts: (a) the farmer must physically **walk** to a tile before working
it, so movement spends turns; (b) crops take days to mature and must be watered;
(c) weeds appear and must be cleared; and (d) there are only **720 turns**, so
efficient routing and a steady plant→water→harvest→sell pipeline matter.

## 2. Architecture

```
        Kaggle env  ──calls──▶  agent(obs)  ──returns──▶  {farmer, hands, market}
                                    │
                                 FarmBrain
                                    │
             ┌──────────────────────┴──────────────────────┐
          protocol.py                                   local_env.py
      (real obs/action shapes)                (local simulator for dev + tests)
```

- **`protocol.py`** — the exact real observation/action contract (constants,
  action builders, coordinate helpers). One source of truth.
- **`local_env.py`** — `LocalKaggricultureEnv`, a deterministic simulator that
  reproduces those shapes so we develop and test **without** the Kaggle package.
  All unconfirmed numbers are isolated in a `Mechanics` dataclass.
- **`brain.py`** — `FarmBrain`: market-first, then nearest
  `harvest → water → weed → plant` target; move-or-act. Pure stdlib.
- **`submission.py` / `kaggriculture_submission.py`** — the top-level `agent(obs)`
  Kaggle calls; the latter is a single self-contained file, parity-tested.

In [ ]:
# Make the package importable (adjust path if running on Kaggle)
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

from kaggriculture_agent.real import FarmBrain, LocalKaggricultureEnv
from kaggriculture_agent.real import protocol as P
print('imports OK')

## 3. One observation in, one action out

The whole agent is a pure function of the observation. Here is a real `obs` from
the local simulator and the action the brain returns for it.

In [ ]:
brain = FarmBrain()
env = LocalKaggricultureEnv(seed=0, n_players=2)
obs = env.observe(0)

print('player   :', obs['player'], '| day:', obs['day'])
print('farmer   :', obs['farms'][0]['farmer'], '| money:', obs['farms'][0]['money'])
print('seeds    :', obs['private']['seeds'], '| shed:', obs['private']['shed'])
print('action   ->', brain.decide(obs))

## 4. Play a full 720-turn game

The agent plays every turn against an idle opponent in the local simulator. Money
climbs as the plant → water → harvest → sell pipeline runs. *(Assumed mechanics,
idle opponent — a sanity check, not a leaderboard score.)*

In [ ]:
env = LocalKaggricultureEnv(seed=0, n_players=2)
brain = FarmBrain()

while not env.done:
    env.step([brain.decide(env.observe(0)), P.pass_action()])
    if env.turn % (env.mech.turns_per_day * 6) == 0:   # every ~6 days
        f = env.farms[0]
        print(f"day {env.day:>2} | money {f['money']:8.2f} | shed {f['shed']}")

print('\nFinal  — our score :', env.score(0))
print('Final  — opponent  :', env.score(1))
print('Turns played       :', env.turn)

## 5. Reliability: the shipped file matches the tested code

The agent ships as one self-contained file (`deliverables/kaggriculture_submission.py`)
with a top-level `agent(obs)`. A parity check confirms that flattened file returns
the **identical** action to the tested package for the same observation — so what
we submit is exactly what we verified.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    'kaggriculture_submission', '../deliverables/kaggriculture_submission.py')
sub = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sub)

env = LocalKaggricultureEnv(seed=3, n_players=2)
brain = FarmBrain()
mismatches = 0
while not env.done:
    o = env.observe(0)
    if sub.agent(o) != brain.decide(o):
        mismatches += 1
    env.step([brain.decide(o), P.pass_action()])

print('submission vs package mismatches:', mismatches, '(expected 0)')
print('sample submission action        :', sub.agent(env.observe(0)))

## 6. Calibrating to the real environment

Our local `Mechanics` (prices, yields, weed rate, scoring) are **assumptions** —
they only affect *offline* practice scores, never leaderboard correctness, because
the real Kaggle env supplies real mechanics and only calls our `agent(obs)`.

`deliverables/MECHANICS_CALIBRATION.md` maps every assumed value to its real source
and the exact field to edit. Reconciling from one real episode is a one-place
change; **the agent code does not change.** Higher-level strategy — e.g. an
optional Gemini planner for crop mix / opponent play — is a documented next step,
reusing the security + guardrail infrastructure already in the repo.

## 7. What I learned & next steps

- **Contract-first** made the agent runnable and fully tested *before* the real
  API was available — and portable into it unchanged.
- **A local simulator** that mirrors the real obs/action shapes turned "we can't
  test until we're on Kaggle" into a fast local dev loop (66 tests).
- **Rule-based, pure-function brain** = cheap, deterministic, explainable, and
  shippable as a single parity-tested file.
- **Honesty as a feature:** every unconfirmed number is labeled and isolated, so
  results are never overstated.
- **Next:** confirm the env name + run/submit path, calibrate `Mechanics` from a
  real episode, add opponent-aware strategy, and A/B an optional Gemini planner.

_Built for the Google × Kaggle AI Agents Intensive — Vibe Coding Capstone._